In [1]:
!nrnivmodl /home/wzl/LFPy/project/conductance_measurement/realistic_neuron/hippocampal_pyramidal_neuron/mod
from neuron import h, gui

# 加载基础的 NEURON GUI 库
h.load_file("nrngui.hoc")

# 依次打开各个 HOC 文件
h.load_file("n128.hoc")               # 几何文件
h.load_file("axon_sections.hoc")      # 轴突部分
h.load_file("basal_dendrite.hoc")     # 基础树突
h.load_file("apical_dendrite.hoc")    # 顶端树突
h.load_file("apical_trunk.hoc")       # 顶端树干
h.load_file("radiatum.hoc")           # 放射区
h.load_file("init.hoc")               # 初始化设置
h.load_file("addgraph.hoc")           # 添加图形显示
from neuron import h
from neuron.units import ms, mV
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import plotly
from neuron import clear_gui_callback
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
from scipy.integrate import cumtrapz
from numpy.linalg import pinv, eig
import numpy as np
from multiprocessing import Pool
font2={'family':'Times New Roman',
'weight':'bold',
'size': 10}
# 自定义刻度标签显示格式

from matplotlib.ticker import FuncFormatter
def format_func(value, tick_number):
    return f'{value:.0f}'  # 将刻度值格式化为小数点后两位

def plot_curve(ax2,x,y,color='black',title='', linestyle='-',xlim=[0,100],dx=20,
               ylim=[0,10],dy=5.0, fontsize=10, alpha = 1.0, xlabel='Time/ms', 
               ylabel = 'Frequency/Hz', label='',ynum_decimals =1, 
               nonscatter=True, s0=10):
    if nonscatter:
        if color:
            ax2.plot(x,y,color=color,linewidth=2,linestyle=linestyle,alpha=alpha, label=label)
        else:
            ax2.plot(x,y,linewidth=2,alpha=alpha, linestyle=linestyle,label=label)
    else:
        if color:
            ax2.scatter(x,y,color=color,linewidth=2,linestyle=linestyle,alpha=alpha, 
                        label=label, s=s0)
        else:
            ax2.scatter(x,y,linewidth=2,alpha=alpha, linestyle=linestyle,label=label,
                        s=s0)
        
    ax2.set_xlabel(xlabel,font2)
    ax2.set_ylabel(ylabel,font2)
    
    # def format_funcx(value, tick_number, num_decimals=xnum_decimals):
    #     if num_decimals==0:
    #         return f'{value:.0f}'
    #     return f'{value:.{num_decimals}f}'

    def format_funcy(value, tick_number, num_decimals=ynum_decimals):
        if num_decimals==0:
          return f'{value:.0f}'
        return f'{value:.{num_decimals}f}'

    # if dx:
    #     ax2.set_xticks(np.arange(xlim[0], xlim[1] + dx, dx))
    #     ax2.set_xticklabels(ax2.get_xticks(), fontsize=fontsize, weight='bold')
    #     ax2.set_xlim([xlim[0], xlim[1]])
    #     ax2.xaxis.set_major_formatter(FuncFormatter(format_funcx))

    if dy:
        ax2.set_yticks(np.arange(ylim[0], ylim[1] + dy, dy))
        ax2.set_yticklabels(ax2.get_yticks(), fontsize=fontsize, weight='bold')
        ax2.set_ylim([ylim[0], ylim[1]])
        ax2.yaxis.set_major_formatter(FuncFormatter(format_funcy))
        
  
    if dx:
       ax2.set_xticks(np.arange(xlim[0],xlim[1]+dx,dx))
       ax2.set_xticklabels(np.arange(xlim[0],xlim[1]+dx,dx),fontsize=10,weight='bold')
       ax2.set_xlim(xlim)
    # if ylim:
    #    ax2.set_yticks(np.arange(ylim[0],ylim[1]+dy,dy))
    #    ax2.set_yticklabels(np.arange(ylim[0],ylim[1]+dy,dy),fontsize=10,weight='bold')
    #    ax2.set_ylim(ylim)
    if title:
       ax2.set_title('{0}'.format(title),fontsize=12,weight='bold')
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    if label:
       ax2.legend(loc='best',prop=font2,edgecolor='white')

/home/wzl/LFPy/project/conductance_measurement/realistic_neuron/hippocampal_pyramidal_neuron/conductance
Mod files: "/home/wzl/LFPy/project/conductance_measurement/realistic_neuron/hippocampal_pyramidal_neuron/mod//home/wzl/LFPy/project/conductance_measurement/realistic_neuron/hippocampal_pyramidal_neuron/mod/h.mod" "/home/wzl/LFPy/project/conductance_measurement/realistic_neuron/hippocampal_pyramidal_neuron/mod//home/wzl/LFPy/project/conductance_measurement/realistic_neuron/hippocampal_pyramidal_neuron/mod/ICaL.mod" "/home/wzl/LFPy/project/conductance_measurement/realistic_neuron/hippocampal_pyramidal_neuron/mod//home/wzl/LFPy/project/conductance_measurement/realistic_neuron/hippocampal_pyramidal_neuron/mod/ICaT.mod" "/home/wzl/LFPy/project/conductance_measurement/realistic_neuron/hippocampal_pyramidal_neuron/mod//home/wzl/LFPy/project/conductance_measurement/realistic_neuron/hippocampal_pyramidal_neuron/mod/kadist.mod" "/home/wzl/LFPy/project/conductance_measurement/realistic_neuron/

--No graphics will be displayed.


Check neuron_morph
	 Note: diam values less than 0.3um are set to 0.3um!
Diam was changed for: 0  points



# estimate time constants|

In [2]:
def exp_fitting(x,y, num=1):
    
    if num==2:
        # Calculate integrals
        iy1 = cumtrapz(y, x, initial=0)
        iy2 = cumtrapz(iy1, x, initial=0)


        # Get exponentials lambdas
        Y = np.column_stack((iy1, iy2,  x**2, x, np.ones_like(x)))
        A = pinv(Y) @ y

        lambdas = eig(np.array([[A[0], A[1]], [1, 0]]))[0]
        # print("Lambdas:", lambdas)

        # Get exponentials multipliers
        X = np.column_stack((np.ones_like(x), np.exp(lambdas[0] * x), np.exp(lambdas[1] * x)))
        P = pinv(X) @ y
        # print("Multipliers:", P)
    
    if num==1:
        iy1 = cumtrapz(y, x, initial=0)

        # Get exponentials lambdas
        Y = np.column_stack((iy1, x, np.ones_like(x)))
        A = pinv(Y) @ y

        lambdas = A[0]
        # print("Lambdas:", lambdas)


        # Get exponentials multipliers
        X = np.column_stack((np.ones_like(x), np.exp(lambdas * x)))
        P = pinv(X) @ y
        # print("Multipliers:", P)
    
    return lambdas, P

def time_constant_fitting(run_dt, dtime, fE, psection=True, passive=False, E = True, loc=58):
    
 
    
    h.init()
    if passive:
        for sec in h.allsec():
            # 检查 section 是否有主动离子通道
            if h.ismembrane("hd", sec=sec):
                # 将电导设置为0
                sec.ghdbar_hd = 0.
            if h.ismembrane("cal", sec=sec):
                sec.gcalbar_cal = 0.
            if h.ismembrane("cat", sec=sec):
                sec.gbar_cat = 0.
            if h.ismembrane("kad", sec=sec):
                sec.gkabar_kad = 0.
            if h.ismembrane("kap", sec=sec):
                sec.gkabar_kap = 0.
            if h.ismembrane("kdr", sec=sec):
                sec.gkdrbar_kdr = 0.
            if h.ismembrane("na3",sec=sec):
                sec.gbar_na3 = 0.
            if h.ismembrane("nax",sec=sec):
                sec.gbar_nax = 0.
    if psection:
        print(h.soma.psection())   
    if passive:
        h.v_init = -71.3
    else:
        h.v_init = -70.        
             
    # Set simulation parameters
    h.dt = run_dt
    h.tstop = 600
    h.v_init = -77.
    v_clamp = -77.


    # Initialize an excitatory or inhibitory synaptic input
    # make a new stimulator
    stim = h.NetStim()

    # attach it to a synapse
    syn = h.Exp2Syn(h.dend[int(loc)](0.9))

    syn.tau1 = 1.  # ms
    syn.tau2 = 5.  # ms
    if E:
        syn.e = 0.  # mV for E input and -100 for I input
    else: 
        syn.e = -100.
    # print(syn.e)
        
    stim.number = 1
    stim.start = h.tstop - 100.  # ms
    ncstim = h.NetCon(stim, syn)
    ncstim.delay = 0.
    ncstim.weight[0] = 0e-2  # uS


    # Initialize voltage clamp
    vclamp = h.SEClamp(h.soma(0.5))
    vclamp.amp1 = v_clamp
    vclamp.dur2 = h.tstop
    vclamp.amp2 = v_clamp - 20.

    irec = h.Vector()
    irec.record(vclamp._ref_i)

    # Function to initialize the simulation
    def initialize():
        h.t = 0
        h.finitialize(h.v_init)
        h.fcurrent()

    # Function to integrate the simulation
    def integrate():
        while h.t < h.tstop:
            h.fadvance()

    # Function to run the simulation
    def go():
        initialize()
        integrate()

    # Main simulation loop
    def main():
        # vol = h.Vector()
    
        # savdata = open("neuron_voltage.dat", "w")
        T = h.tstop/run_dt+1
        N = int(10./dtime)
        data = np.zeros((2*N, int(T)))


        for i in range(1, N+1):
            vclamp.dur1 = stim.start + i * dtime

            ncstim.weight[0] = 0e-2
            go()
            data[2*(i-1),:] = irec.to_python()  # Assuming irec is a NEURON Vector

            
            ncstim.weight[0] = fE
            go()
            data[2*(i-1)+1,:] = irec.to_python()  # Assuming irec is a NEURON Vector

        I_w_ng = data[::2, :]
        I_w_g = data[1::2, :]
        diff_I = I_w_ng - I_w_g
        x = np.arange(0.1, 10.1, dtime) # ms
        y = -np.sum(diff_I, axis=1)  # size of 100

        result_lambdas, result_multipliers = exp_fitting(x, y, num=2)
        # ground_truth = np.array([-1., -0.2])
        # relative_error = np.abs((result_lambdas - ground_truth)/ground_truth)
        return result_lambdas
    
    
    result_lambdas = main()
    return result_lambdas

In [4]:
# location
def recovery_of_time_const_parallel(args,):
    loc0, = args
    result_lambdas = time_constant_fitting(
            0.1, 0.1, 6e-4, passive = True, E = True,  loc = loc0, psection=False)
     
    return result_lambdas

loc = np.arange(0,199,1)

# Create a Pool with desired number of processes (adjust as needed)
num_processes = 50
with Pool(num_processes) as pool:
    # Prepare arguments for parallel processing
    args_list = [(loc0,) for loc0 in loc]

    # Apply the function in parallel
    results = pool.map(recovery_of_time_const_parallel, args_list)

# Process the results
est_time_const = np.zeros((len(loc), 2))
for i, result in enumerate(results):
    est_time_const[i, :] = result
    
np.savetxt("est_time_const_passive_dend199_fE6e4.txt", est_time_const)

In [5]:
# location
def recovery_of_time_const_parallel(args,):
    loc0, = args
    result_lambdas = time_constant_fitting(
            0.1, 0.1, 6e-4, passive = True, E = False,  loc = loc0, psection=False)
     
    return result_lambdas

loc = np.arange(0,199,1)

# Create a Pool with desired number of processes (adjust as needed)
num_processes = 50
with Pool(num_processes) as pool:
    # Prepare arguments for parallel processing
    args_list = [(loc0,) for loc0 in loc]

    # Apply the function in parallel
    results = pool.map(recovery_of_time_const_parallel, args_list)

# Process the results
est_time_const = np.zeros((len(loc), 2))
for i, result in enumerate(results):
    est_time_const[i, :] = result
    
np.savetxt("est_time_const_passive_dend199_fI6e4.txt", est_time_const)

In [6]:
# location
def recovery_of_time_const_parallel(args,):
    loc0, = args
    result_lambdas = time_constant_fitting(
            0.1, 0.1, 6e-4, passive = False, E = True,  loc = loc0, psection=False)
     
    return result_lambdas

loc = np.arange(0,199,1)

# Create a Pool with desired number of processes (adjust as needed)
num_processes = 50
with Pool(num_processes) as pool:
    # Prepare arguments for parallel processing
    args_list = [(loc0,) for loc0 in loc]

    # Apply the function in parallel
    results = pool.map(recovery_of_time_const_parallel, args_list)

# Process the results
est_time_const = np.zeros((len(loc), 2))
for i, result in enumerate(results):
    est_time_const[i, :] = result
    
np.savetxt("est_time_const_active_dend199_fE6e4.txt", est_time_const)

/tmp/ipykernel_2570528/3564092593.py:23: ComplexWarning: Casting complex values to real discards the imaginary part
  est_time_const[i, :] = result


In [7]:
# location
def recovery_of_time_const_parallel(args,):
    loc0, = args
    result_lambdas = time_constant_fitting(
            0.1, 0.1, 6e-4, passive = False, E = False,  loc = loc0, psection=False)
     
    return result_lambdas

loc = np.arange(0,199,1)

# Create a Pool with desired number of processes (adjust as needed)
num_processes = 50
with Pool(num_processes) as pool:
    # Prepare arguments for parallel processing
    args_list = [(loc0,) for loc0 in loc]

    # Apply the function in parallel
    results = pool.map(recovery_of_time_const_parallel, args_list)

# Process the results
est_time_const = np.zeros((len(loc), 2))
for i, result in enumerate(results):
    est_time_const[i, :] = result
    
np.savetxt("est_time_const_active_dend199_fI6e4.txt", est_time_const)

/tmp/ipykernel_2570528/2394882680.py:23: ComplexWarning: Casting complex values to real discards the imaginary part
  est_time_const[i, :] = result


In [8]:
# strength
def recovery_of_time_const_parallel(args,):
    fE0, = args
    result_lambdas = time_constant_fitting(
            0.1, 0.1, fE0, passive = True, E = True,  loc = 40, psection=False)
   
     
    return result_lambdas

# EPSP: 0.5mV - 5.6mV
# IPSP: 0.2mV - 0.6mV

fE = np.linspace(0.5e-3, 4e-3, 50)

# Create a Pool with desired number of processes (adjust as needed)
num_processes = 50
with Pool(num_processes) as pool:
    # Prepare arguments for parallel processing
    args_list = [(fE0,) for fE0 in fE]

    # Apply the function in parallel
    results = pool.map(recovery_of_time_const_parallel, args_list)

# Process the results
est_time_const = np.zeros((len(fE), 2))
for i, result in enumerate(results):
    est_time_const[i, :] = result
    
np.savetxt("est_time_const_passive_loc40_fE.txt", est_time_const)

In [9]:
# strength
def recovery_of_time_const_parallel(args,):
    fE0, = args
    result_lambdas = time_constant_fitting(
            0.1, 0.1, fE0, passive = True, E = False,  loc = 40, psection=False)
   
     
    return result_lambdas

# EPSP: 0.5mV - 5.6mV
# IPSP: 0.2mV - 0.6mV

fE = np.linspace(0.5e-3, 4e-3, 50)

# Create a Pool with desired number of processes (adjust as needed)
num_processes = 50
with Pool(num_processes) as pool:
    # Prepare arguments for parallel processing
    args_list = [(fE0,) for fE0 in fE]

    # Apply the function in parallel
    results = pool.map(recovery_of_time_const_parallel, args_list)

# Process the results
est_time_const = np.zeros((len(fE), 2))
for i, result in enumerate(results):
    est_time_const[i, :] = result
    
np.savetxt("est_time_const_passive_loc40_fI.txt", est_time_const)

In [7]:
result_lambdas = time_constant_fitting(
            0.1, 0.1, 6e-4, passive = False, E = True)

{'point_processes': {}, 'density_mechs': {'pas': {'g': [1.6666666666666667e-05], 'e': [-71.64064333062406], 'i': [2.7344055510400978e-05]}, 'hd': {'ghdbar': [2e-06], 'vhalfl': [-73.0], 'i': [-3.2586672003674416e-05], 'l': [0.40733340004593027]}, 'kap': {'gkabar': [0.0005], 'gka': [1.9908635918562848e-07], 'n': [0.00047964919017608594], 'l': [0.8301332026122721]}, 'kdr': {'gkdrbar': [0.005], 'gkdr': [4.110698979065775e-07], 'n': [8.22139795813155e-05]}, 'na3': {'gbar': [0.03], 'ar2': [1.0], 'm': [0.012317107055633518], 'h': [0.9933071490757153], 's': [1.0]}}, 'ions': {'na': {'ena': [55.0], 'nai': [10.0], 'nao': [140.0], 'ina': [-6.960508648570605e-06], 'dina_dv_': [5.568406918816823e-08]}, 'k': {'ek': [-90.0], 'ki': [54.4], 'ko': [2.5], 'ik': [1.2203125141844119e-05], 'dik_dv_': [6.101562570952918e-07]}}, 'morphology': {'L': 0.0099945068359375, 'diam': [18.079999923706055], 'pts3d': [(-0.6890000104904175, 7.706999778747559, 234.48500061035156, 18.079999923706055), (-0.6890000104904175, 

KeyboardInterrupt: 